In [1]:
import numpy as np
from scipy.linalg import solve
from scipy.stats import norm
from scipy.integrate import quad
import matplotlib.pyplot as plt

In [2]:
def gbm_rates_simulator(S0, r_func, sigma, T, Mt, Ms):
    R, _ = quad(r_func, 0, T)
    Z = np.random.normal(size=Ms)
    S_T = S0 * np.exp(
        R - 0.5 * sigma**2 * T + sigma * np.sqrt(T) * Z
    )
    return S_T

def put_price_closed_form(S0, K, T, sigma, r_func):
    dt = 1e-5
    t_grid = np.arange(0, T, dt)
    R = np.trapz([r_func(t) for t in t_grid], dx=dt)
    d1 = (np.log(S0 / K) + R + 0.5 * sigma**2 * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    put = K * np.exp(-R) * norm.cdf(-d2) - S0 * norm.cdf(-d1)
    return put

def matrix_L_bs(Ns, S_grid, r_t, sigma):
    dS = S_grid[1] - S_grid[0]
    def L_disc():
        return -np.diag(r_t * np.ones(Ns))
    def L_S():
        L = np.zeros((Ns, Ns))
        for i in range(1, Ns - 1):
            drift = r_t * S_grid[i]
            L[i, i - 1] = -drift / (2 * dS)
            L[i, i + 1] =  drift / (2 * dS)
        return L
    def L_SS():
        L = np.zeros((Ns, Ns))
        for i in range(1, Ns - 1):
            coeff = 0.5 * sigma**2 * S_grid[i]**2 / dS**2
            L[i, i - 1] = coeff
            L[i, i]     = -2 * coeff
            L[i, i + 1] = coeff
        return L
    return L_disc() + L_S() + L_SS()

def european_put_cn(S0, K, r_func, sigma, T, Nt=250, Ns=150, Smax=None):
    if Smax is None:
        Smax = max(3 * K, 5 * S0)
    S_grid = np.linspace(0, Smax, Ns)
    dS = S_grid[1] - S_grid[0]
    dt = T / Nt
    # terminal condition (put payoff)
    V = np.maximum(K - S_grid, 0)
    I = np.eye(Ns)
    for n in reversed(range(Nt)):
        tau = n * dt
        r_tau = r_func(T - tau)
        L = matrix_L_bs(Ns, S_grid, r_tau, sigma)
        M1 = I - 0.5 * dt * L
        M2 = I + 0.5 * dt * L
        V = solve(M1, M2 @ V)
        V[0] = K * np.exp(-r_func(T) * (T - tau))   # S=0 intrinsic
        V[-1] = 0                                   # S→∞ payoff → 0
    return np.interp(S0, S_grid, V)

In [3]:
# Parameters
S0 = 10
K = 10
T = 1
sigma = 0.2
Mt, Ms = 1000, 10000
Nt, Ns = 250, 150

r_func = lambda t: 0.05 + 0.095 * t**2  # example time-dependent rate

ST = gbm_rates_simulator(S0, r_func, sigma, T, Mt, Ms)

In [4]:
price = put_price_closed_form(S0, K, T, sigma, r_func)
print(price)

0.435902556116917


In [5]:
put_price = np.mean(np.maximum(K - ST, 0))
print(put_price)

0.47127083457981533


In [6]:
european_put_cn(S0, K, r_func, sigma, T)

0.4355292728917105